# 📊 SHS Survey Data Analysis - Plotly Interactive Edition
### Comprehensive analysis with interactive visualizations exported as HTML & PNG

In [ ]:
# 🔧 IMPORTS & SETUP
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

# Plotly imports
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistical tests
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

# For saving Plotly as PNG (requires kaleido)
# Install with: pip install -U kaleido
try:
    import kaleido
    KALEIDO_AVAILABLE = True
except ImportError:
    KALEIDO_AVAILABLE = False
    print("⚠️  kaleido not installed. PNG export will be skipped. Run: pip install -U kaleido")

# Create output directory
os.makedirs('SHS_outputs', exist_ok=True)
os.makedirs('SHS_outputs/plots_html', exist_ok=True)
os.makedirs('SHS_outputs/plots_png', exist_ok=True)

print("✅ Setup complete!")

In [ ]:
# 📥 LOAD & PREPARE DATA
df_raw = pd.read_csv('SHS_raw.csv')
print(f"Raw data shape: {df_raw.shape}")

# Filter for completed responses
df = df_raw[(df_raw['Finished'] == True) & (df_raw['Condition'] != 'NA')].copy()
print(f"After filtering completed & valid condition: {df.shape}")

# Clean emotion columns: extract numeric value from "5 (Very)" -> 5, "1 (Not at all)" -> 1
emotion_cols_neg = [c for c in df.columns if 'EmotionNegative' in c]
emotion_cols_pos = [c for c in df.columns if 'EmotionPositive' in c]
action_cols = [c for c in df.columns if c.startswith('Actions_')]
political_cols = [c for c in df.columns if c.startswith('PoliticalActions_')]

def clean_likert(series):
    """Extract numeric value from Likert scale strings"""
    return series.astype(str).str.extract(r'(\d+)', expand=False).astype(float)

# Apply cleaning
for col in emotion_cols_neg + emotion_cols_pos:
    if col in df.columns:
        df[col + '_num'] = clean_likert(df[col])

# Create composite scores
neg_cols_num = [c + '_num' for c in emotion_cols_neg if c + '_num' in df.columns]
pos_cols_num = [c + '_num' for c in emotion_cols_pos if c + '_num' in df.columns]

df['EmotionNegative_Score'] = df[neg_cols_num].mean(axis=1, skipna=True)
df['EmotionPositive_Score'] = df[pos_cols_num].mean(axis=1, skipna=True)
df['Emotion_Balance'] = df['EmotionPositive_Score'] - df['EmotionNegative_Score']

# Clean action columns for analysis
action_map = {
    'Extremely likely': 5, 'Somewhat likely': 4, 'Neither likely nor unlikely': 3,
    'Somewhat unlikely': 2, 'Extremely unlikely': 1
}

for col in action_cols + political_cols:
    if col in df.columns:
        df[col + '_num'] = df[col].map(action_map)

# Donation variable: combine Bird/Wild/Bio into single indicator
df['Donation_Any'] = df[['DonationBird', 'DonationWild', 'DonationBio']].eq('Yes').any(axis=1).astype(int)
df['Donation_Count'] = df[['DonationBird', 'DonationWild', 'DonationBio']].eq('Yes').sum(axis=1)

# Save cleaned data
df.to_csv('SHS_outputs/SHS_cleaned.csv', index=False)
print("✅ Data cleaning complete. Saved to SHS_outputs/SHS_cleaned.csv")

In [ ]:
# 📊 HELPER FUNCTION: Save Plotly figures as HTML + PNG
def save_plotly_fig(fig, filename_base, title="", width=1000, height=600):
    """Save a Plotly figure as both HTML and PNG"""
    # Update layout for better exports
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor='center'),
        width=width,
        height=height,
        template='plotly_white',
        hovermode='x unified'
    )
    
    # Save as HTML (always works)
    html_path = f'SHS_outputs/plots_html/{filename_base}.html'
    fig.write_html(html_path, include_plotlyjs='cdn')
    print(f"✅ Saved HTML: {html_path}")
    
    # Save as PNG (requires kaleido)
    if KALEIDO_AVAILABLE:
        png_path = f'SHS_outputs/plots_png/{filename_base}.png'
        fig.write_image(png_path, scale=2)  # 2x resolution for print quality
        print(f"✅ Saved PNG: {png_path}")
    else:
        print(f"⚠️  Skipping PNG export for {filename_base} (install kaleido for PNG support)")
    
    return fig

In [ ]:
# 📈 PLOT 1: Condition Distribution (Bar Chart)
condition_counts = df['Condition'].value_counts().reset_index()
condition_counts.columns = ['Condition', 'Count']

fig1 = px.bar(
    condition_counts, x='Condition', y='Count',
    color='Condition',
    title='📋 Response Count by Experimental Condition',
    labels={'Count': 'Number of Responses', 'Condition': 'Condition'},
    text='Count'
)
fig1.update_traces(textposition='outside')
save_plotly_fig(fig1, '01_condition_distribution', 'Response Count by Condition')
fig1.show()

In [ ]:
# 📈 PLOT 2: Emotion Scores by Condition (Box Plot - Interactive)
df_melted_emotion = df.melt(
    id_vars=['ResponseId', 'Condition'],
    value_vars=['EmotionNegative_Score', 'EmotionPositive_Score', 'Emotion_Balance'],
    var_name='Score_Type',
    value_name='Score'
)

fig2 = px.box(
    df_melted_emotion,
    x='Condition', y='Score', color='Score_Type',
    title='😊 Emotion Scores by Condition',
    labels={'Score': 'Average Likert Score (1-5)', 'Condition': 'Experimental Condition'},
    points='all', hover_data=['ResponseId']
)
fig2.update_layout(legend_title_text='Score Type')
save_plotly_fig(fig2, '02_emotion_boxplot', 'Emotion Scores by Condition')
fig2.show()

In [ ]:
# 📈 PLOT 3: Donation Proportions by Condition (Grouped Bar)
donation_by_condition = df.groupby('Condition')['Donation_Any'].agg(['sum', 'count']).reset_index()
donation_by_condition['Donation_Rate'] = donation_by_condition['sum'] / donation_by_condition['count'] * 100

fig3 = px.bar(
    donation_by_condition,
    x='Condition', y='Donation_Rate',
    color='Condition',
    title='💰 Donation Rate (%) by Condition',
    labels={'Donation_Rate': 'Donation Rate (%)', 'Condition': 'Condition'},
    text=donation_by_condition['Donation_Rate'].round(1).astype(str) + '%'
)
fig3.update_traces(textposition='outside')
fig3.update_yaxes(range=[0, 110])
save_plotly_fig(fig3, '03_donation_rate', 'Donation Rate by Condition')
fig3.show()

In [ ]:
# 📈 PLOT 4: Emotion Score Distributions (Histogram with KDE)
fig4 = make_subplots(rows=1, cols=3, subplot_titles=['Negative Emotions', 'Positive Emotions', 'Emotion Balance'])

# Negative emotions histogram
fig4.add_trace(
    go.Histogram(x=df['EmotionNegative_Score'], nbinsx=10, name='Negative', marker_color='salmon', opacity=0.7),
    row=1, col=1
)

# Positive emotions histogram
fig4.add_trace(
    go.Histogram(x=df['EmotionPositive_Score'], nbinsx=10, name='Positive', marker_color='lightgreen', opacity=0.7),
    row=1, col=2
)

# Emotion balance histogram
fig4.add_trace(
    go.Histogram(x=df['Emotion_Balance'], nbinsx=15, name='Balance', marker_color='skyblue', opacity=0.7),
    row=1, col=3
)

fig4.update_layout(
    title='📊 Distribution of Emotion Scores',
    showlegend=False,
    height=400
)
fig4.update_xaxes(title_text='Score (1-5)', row=1, col=1)
fig4.update_xaxes(title_text='Score (1-5)', row=1, col=2)
fig4.update_xaxes(title_text='Balance (Pos - Neg)', row=1, col=3)
fig4.update_yaxes(title_text='Count', row=1, col=1)

save_plotly_fig(fig4, '04_emotion_distributions', 'Emotion Score Distributions', height=500)
fig4.show()

In [ ]:
# 📈 PLOT 5: Action Likelihood by Condition (Heatmap-style Bar)
action_numeric_cols = [c + '_num' for c in action_cols if c + '_num' in df.columns]
action_labels = [c.replace('Actions_', '').replace('_num', '') for c in action_numeric_cols]

action_by_condition = df.groupby('Condition')[action_numeric_cols].mean().T
action_by_condition.index = action_labels

fig5 = go.Figure(data=go.Heatmap(
    z=action_by_condition.values,
    x=action_by_condition.columns,
    y=action_by_condition.index,
    colorscale='Viridis',
    text=action_by_condition.round(2).values,
    texttemplate='%{text}',
    textfont={"size": 10},
    hoverongaps=False
))

fig5.update_layout(
    title='🎯 Average Action Likelihood by Condition (1=Unlikely, 5=Likely)',
    xaxis_title='Condition',
    yaxis_title='Action Item',
    height=400
)

save_plotly_fig(fig5, '05_actions_heatmap', 'Action Likelihood Heatmap', width=1100, height=500)
fig5.show()

In [ ]:
# 📈 PLOT 6: Correlation Matrix (Interactive Heatmap)
numeric_cols = ['Age', 'Duration (in seconds)', 'EmotionNegative_Score', 'EmotionPositive_Score', 
                'Emotion_Balance', 'Donation_Count'] + [c for c in df.columns if 'Actions' in c and '_num' in c]
numeric_cols = [c for c in numeric_cols if c in df.columns and df[c].notna().any()]

corr_matrix = df[numeric_cols].corr()

fig6 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=corr_matrix.round(2).values,
    texttemplate='%{text}',
    textfont={"size": 9},
    hoverongaps=False
))

fig6.update_layout(
    title='🔗 Correlation Matrix of Key Variables',
    height=700,
    width=800
)

save_plotly_fig(fig6, '06_correlation_matrix', 'Variable Correlations', height=800, width=900)
fig6.show()

In [ ]:
# 📈 PLOT 7: Duration vs Emotion Balance (Scatter with Condition Color)
fig7 = px.scatter(
    df,
    x='Duration (in seconds)', y='Emotion_Balance',
    color='Condition',
    size='Donation_Count',
    hover_data=['ResponseId', 'Age', 'Gender'],
    title='⏱️ Survey Duration vs Emotion Balance',
    labels={'Duration (in seconds)': 'Duration (seconds)', 'Emotion_Balance': 'Emotion Balance (Positive - Negative)'}
)
fig7.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Neutral Balance")
save_plotly_fig(fig7, '07_duration_vs_emotion', 'Duration vs Emotion Balance', width=1000, height=600)
fig7.show()

In [ ]:
# 🧪 HYPOTHESIS TESTING RESULTS (Text Output + Summary Table)
results = {}

# H1: Emotion scores differ by condition (ANOVA)
print("\n" + "="*60)
print("🧪 HYPOTHESIS TESTING RESULTS")
print("="*60)

# H1: One-way ANOVA for Emotion_Balance by Condition
print("\n📋 H1: Emotion Balance differs by Condition")
anova_model = ols('Emotion_Balance ~ C(Condition)', data=df).fit()
anova_table = sm.stats.anova_lm(anova_model, typ=2)
print(anova_table)
results['H1_ANOVA'] = {
    'F_stat': float(anova_table['F']['C(Condition)']),
    'p_value': float(anova_table['PR(>F)']['C(Condition)']),
    'significant': float(anova_table['PR(>F)']['C(Condition)']) < 0.05
}

# Post-hoc Tukey if ANOVA significant
if results['H1_ANOVA']['significant']:
    tukey = pairwise_tukeyhsd(endog=df['Emotion_Balance'], groups=df['Condition'], alpha=0.05)
    print("\n🔍 Tukey HSD Post-hoc:")
    print(tukey)
    results['H1_Tukey'] = str(tukey)

# H2: Chi-square for Donation by Condition
print("\n📋 H2: Donation behavior differs by Condition")
contingency = pd.crosstab(df['Condition'], df['Donation_Any'])
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"Chi-square: {chi2:.3f}, p-value: {p:.4f}, df: {dof}")
print("\nContingency Table:")
print(contingency)
results['H2_ChiSquare'] = {
    'chi2': float(chi2),
    'p_value': float(p),
    'significant': p < 0.05
}

# H3: Emotion balance predicts donations (Mann-Whitney U)
print("\n📋 H3: Emotion Balance differs between Donors vs Non-Donors")
donors = df[df['Donation_Any'] == 1]['Emotion_Balance']
non_donors = df[df['Donation_Any'] == 0]['Emotion_Balance']
u_stat, p_mw = mannwhitneyu(donors, non_donors, alternative='two-sided')
print(f"Mann-Whitney U: {u_stat:.1f}, p-value: {p_mw:.4f}")
print(f"Donors mean balance: {donors.mean():.2f} | Non-donors: {non_donors.mean():.2f}")
results['H3_MannWhitney'] = {
    'U_stat': float(u_stat),
    'p_value': float(p_mw),
    'significant': p_mw < 0.05
}

# Save results summary
with open('SHS_outputs/SHS_hypothesis_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"\n✅ Results saved to SHS_outputs/SHS_hypothesis_results.json")

In [ ]:
# 📈 PLOT 8: Hypothesis Results Summary (Interactive Table)
results_summary = []
for h, res in results.items():
    if isinstance(res, dict) and 'p_value' in res:
        results_summary.append({
            'Hypothesis': h,
            'Test': res.get('F_stat', res.get('chi2', res.get('U_stat', 'N/A'))),
            'p-value': f"{res['p_value']:.4f}",
            'Significant (α=0.05)': '✅ Yes' if res['significant'] else '❌ No'
        })

results_df = pd.DataFrame(results_summary)

fig8 = go.Figure(data=[go.Table(
    header=dict(values=list(results_df.columns),
                fill_color='royalblue', font_color='white', align='left'),
    cells=dict(values=[results_df[col] for col in results_df.columns],
               fill_color='lavender', align='left'))
])

fig8.update_layout(
    title='📋 Hypothesis Testing Summary',
    height=300
)

save_plotly_fig(fig8, '08_results_summary', 'Hypothesis Results', height=400)
fig8.show()

In [ ]:
# 📈 PLOT 9: Logistic Regression - Feature Importance (Bar Chart)
print("\n📋 H5: Logistic Regression - Predicting Donation Behavior")

# Prepare features
features = ['Age', 'EmotionNegative_Score', 'EmotionPositive_Score', 'Emotion_Balance', 'Duration (in seconds)']
features = [f for f in features if f in df.columns and df[f].notna().sum() > 10]

# Encode Condition as dummy variables
df_model = df[features + ['Condition', 'Donation_Any']].dropna()
df_model = pd.get_dummies(df_model, columns=['Condition'], drop_first=True)

X = df_model.drop('Donation_Any', axis=1)
y = df_model['Donation_Any']

# Fit model
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X, y)

# Cross-validation score
cv_scores = cross_val_score(log_reg, X, y, cv=5)
print(f"Cross-validation accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std()*2:.3f})")

# Feature importance plot
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

fig9 = px.bar(
    coef_df,
    x='Coefficient', y='Feature',
    color='Coefficient',
    color_continuous_scale='RdBu',
    title='🎯 Logistic Regression: Feature Importance for Donation Prediction',
    labels={'Coefficient': 'Model Coefficient', 'Feature': 'Predictor'},
    orientation='h'
)
fig9.add_vline(x=0, line_dash="dash", line_color="gray")
save_plotly_fig(fig9, '09_logistic_features', 'Logistic Regression Coefficients', width=900, height=500)
fig9.show()

results['H5_LogisticRegression'] = {
    'cv_accuracy_mean': float(cv_scores.mean()),
    'cv_accuracy_std': float(cv_scores.std()),
    'top_features': coef_df.head(5).to_dict('records')
}

In [ ]:
# 📈 PLOT 10: Interactive Dashboard-Style Summary (Subplots)
fig10 = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Donation Rate by Condition', 'Emotion Balance Distribution',
                    'Duration vs Balance', 'Action Likelihood Summary'],
    specs=[[{"type": "bar"}, {"type": "histogram"}],
           [{"type": "scatter"}, {"type": "bar"}]]
)

# Top-left: Donation rate
fig10.add_trace(
    go.Bar(x=donation_by_condition['Condition'], y=donation_by_condition['Donation_Rate'],
           marker_color='gold', text=donation_by_condition['Donation_Rate'].round(1).astype(str)+'%',
           textposition='auto'),
    row=1, col=1
)

# Top-right: Emotion balance histogram
fig10.add_trace(
    go.Histogram(x=df['Emotion_Balance'], nbinsx=12, marker_color='lightblue', opacity=0.8),
    row=1, col=2
)

# Bottom-left: Duration vs balance scatter (one trace per condition so colors are valid + legend works)
condition_color_map = {'Pessimistic': '#d62728', 'Optimistic': '#2ca02c',
                       'Combined': '#1f77b4', 'Control': '#7f7f7f'}
for cond, sub in df.groupby('Condition'):
    fig10.add_trace(
        go.Scatter(
            x=sub['Duration (in seconds)'], y=sub['Emotion_Balance'],
            mode='markers', name=cond,
            marker=dict(color=condition_color_map.get(cond, '#999999'), size=8, opacity=0.7),
            showlegend=True
        ),
        row=2, col=1
    )

# Bottom-right: Average action likelihood
avg_actions = df[action_numeric_cols].mean().reset_index()
avg_actions.columns = ['Action', 'Avg_Likelihood']
avg_actions['Action'] = avg_actions['Action'].str.replace('_num', '').str.replace('Actions_', '')
fig10.add_trace(
    go.Bar(x=avg_actions['Action'], y=avg_actions['Avg_Likelihood'],
           marker_color='lightgreen', text=avg_actions['Avg_Likelihood'].round(2), textposition='auto'),
    row=2, col=2
)

fig10.update_layout(
    title='🎛️ SHS Survey Analysis Dashboard',
    height=800,
    showlegend=True
)
fig10.update_xaxes(title_text='Condition', row=1, col=1)
fig10.update_yaxes(title_text='Donation Rate (%)', row=1, col=1)
fig10.update_xaxes(title_text='Emotion Balance', row=1, col=2)
fig10.update_yaxes(title_text='Count', row=1, col=2)
fig10.update_xaxes(title_text='Duration (seconds)', row=2, col=1)
fig10.update_yaxes(title_text='Emotion Balance', row=2, col=1)
fig10.update_xaxes(title_text='Action', row=2, col=2)
fig10.update_yaxes(title_text='Avg Likelihood (1-5)', row=2, col=2)

save_plotly_fig(fig10, '10_analysis_dashboard', 'SHS Analysis Dashboard', width=1200, height=900)
fig10.show()

In [ ]:
# 💾 FINAL EXPORT SUMMARY
print("\n" + "🎉"*30)
print("✅ ANALYSIS COMPLETE!")
print("🎉"*30)

print(f"""
📁 Output Files Generated:
├── SHS_outputs/
│   ├── SHS_cleaned.csv                    # Preprocessed dataset
│   ├── SHS_hypothesis_results.json        # Statistical test results
│   ├── plots_html/                        # Interactive HTML plots
│   │   ├── 01_condition_distribution.html
│   │   ├── 02_emotion_boxplot.html
│   │   ├── 03_donation_rate.html
│   │   ├── 04_emotion_distributions.html
│   │   ├── 05_actions_heatmap.html
│   │   ├── 06_correlation_matrix.html
│   │   ├── 07_duration_vs_emotion.html
│   │   ├── 08_results_summary.html
│   │   ├── 09_logistic_features.html
│   │   └── 10_analysis_dashboard.html
│   └── plots_png/                         # Static PNG exports (if kaleido installed)
│       ├── 01_condition_distribution.png
│       ├── 02_emotion_boxplot.png
│       └── ... (all plots)

🔍 Key Findings Summary:
""")

for h, res in results.items():
    if isinstance(res, dict) and 'p_value' in res:
        sig = "✅ Significant" if res['significant'] else "❌ Not significant"
        print(f"• {h}: p={res['p_value']:.4f} → {sig}")

print(f"""
💡 Tips:
• Open HTML files in any browser for interactive exploration
• Hover over data points for detailed values
• Use plot controls to zoom, pan, and export snapshots
• Install kaleido for high-quality PNG exports: pip install -U kaleido

🚀 Next Steps:
• Customize visualizations by modifying Plotly parameters
• Add more hypotheses or subgroup analyses
• Export dashboard to PDF using browser print function
""")